# 🚀 Welcome to Your ADK Adventure - MultiAgents! 🚀

Welcome back, Agent Architect! This notebook dives into the heart of the Google Agent Development Kit (ADK): orchestrating teams of specialized agents to tackle complex, multi-step problems that a single agent cannot handle alone.

By the end of this session, you will be an expert in advanced agentic workflows:

- **SequentialAgent**: You'll learn to chain agents together, creating pipelines where the output of one agent becomes the input for the next.

- **LoopAgent**: You'll build iterative systems where agents can plan, critique, and refine their work until a specific goal is met, making them "perfectionists."

- **ParallelAgent**: You'll unleash efficiency by running multiple agents simultaneously and then synthesizing their collective findings into a single, comprehensive answer.

- **The Router**: You will construct a "master" router agent that intelligently analyzes a user's request and delegates it to the correct agent or workflow.

Let's get this adventure started!

## Author

HI, I'm Qingyue (Annie) Wang, a developer advocate and AI engineer at **Google**, passionate about helping developers build with AI and cloud technologies :)


If you have questions with this notebook, contact me on [LinkedIn](https://www.linkedin.com/in/anniewangtech/) , [X](https://twitter.com/anniewangtech) or email anniewangtech0510@Gmail.com


```

  (\__/)
  (•ㅅ•)
  /づ  📚       Enjoy learning AI Agents :)


```


-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: https://codelabs.developers.google.com/onramp/instructions#0

 -----------------------------------------------------------------------------

```
 ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )             
```


## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [1]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4

import vertexai
from google.colab import auth
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent, SequentialAgent, LoopAgent, ParallelAgent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search, ToolContext
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part

print("✅ All libraries are ready to go!")

✅ All libraries are ready to go!


### Authenticate and Configure Your Project
To use Vertex AI, you need an active Google Cloud project. This section handles authenticating your environment and setting up the necessary project configurations.

In [2]:
# ---  Authentication & Project Configuration ---

# Authenticate user in Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()
    print("✅ Authenticated successfully.")

✅ Authenticated successfully.


In [3]:
# @title Set Your Google Cloud Project Details
PROJECT_ID = "project-999ca28f-35e8-45ea-8b7"              # @param {type:"string"}
LOCATION = "us-central1"               # @param {type:"string"}

# Set environment variables for the ADK and gcloud
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

print(f"\n✅ Vertex AI configured for project '{PROJECT_ID}' in '{LOCATION}'.")


✅ Vertex AI configured for project 'project-999ca28f-35e8-45ea-8b7' in 'us-central1'.


In [4]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

---
## Part 1: Multi-Agent Mayhem - Sequential Workflows 🧠→🤖→🤖

You've mastered single agents and memory. Now for the most advanced topic: making agents **work together in a sequence**.

Some tasks are too complex for one agent. A user might ask, "Find me a great restaurant and then tell me how to get there." This requires two different skills: food recommendation and navigation.

We'll build a system that can handle this by:
1.  Creating a new `transportation_agent` 🚗.
2.  Teaching our `router_agent` 🧠 to recognize these special "combo" requests.
3.  Writing Python code (the "orchestrator") that runs the agents in a sequence, passing the output of the first agent to the second.

```
                    +---------------------+
                    |    User Query 🗣️     |
                    +----------+----------+
                               |
                               v
                    +---------------------+
                    |   Router Agent 🤖    |
                    |  (Classify Request) |
                    +----------+----------+
                               |
          +--------------------+----------------------+
          |                    |                      |
          v                    v                      v
  +----------------+   +--------------------+  +----------------------+
  |  foodie_agent  |   | weekend_guide_agent|  |  day_trip_agent      |
  |  🍣 Food Search |   | 🎉 Event Discovery |  | 🧳 Trip Planner       |
  +----------------+   +--------------------+  +----------------------+
          |
          v
  +----------------------------+            (if combo request)
  |  Restaurant Recommendation |---------------------------+
  |  ex: "Best sushi is at X"  |                           |
  +----------------------------+                           v
                                                        +-----------------------+
                                                        | transportation_agent  |
                                                        | 🚗 Get Directions      |
                                                        +-----------------------+
                                                        | Input: origin, place  |
                                                        | Output: directions    |
                                                        +-----------------------+

Final Output: 🍽️ Recommendation + 🚗 Route Info
```


In [5]:
# --- Agent Definitions for our Specialist Team ---
# --- Agent Definition ---

day_trip_agent = Agent(
    name="day_trip_agent",
    model="gemini-2.5-flash",
    description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
    instruction="""
    You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

    Your Mission:
    Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

    Guidelines:
    1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
    2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
    3. **Real-Time Focus**: Search for current operating hours and special events.
    4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

    RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
    """,
    tools=[google_search]
)

foodie_agent = Agent(
    name="foodie_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="You are an expert food critic. Your goal is to find the absolute best food, restaurants, or culinary experiences based on a user's request. When you recommend a place, state its name clearly. For example: 'The best sushi is at **Jin Sho**.'"
)

weekend_guide_agent = Agent(
    name="weekend_guide_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="You are a local events guide. Your task is to find interesting events, concerts, festivals, and activities happening on a specific weekend."
)

transportation_agent = Agent(
    name="transportation_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="You are a navigation assistant. Given a starting point and a destination, provide clear directions on how to get from the start to the end."
)

# --- The Brain of the Operation: The Router Agent ---
# We update the router's instructions to know about the new 'combo' task.
router_agent = Agent(
    name="router_agent",
    model="gemini-2.5-flash",
    instruction="""
    You are a request router. Your job is to analyze a user's query and decide which of the following agents or workflows is best suited to handle it.
    Do not answer the query yourself, only return the name of the most appropriate choice.

    Available Options:
    - 'foodie_agent': For queries *only* about food, restaurants, or eating.
    - 'weekend_guide_agent': For queries about events, concerts, or activities happening on a specific timeframe like a weekend.
    - 'day_trip_agent': A general planner for any other day trip requests.
    - 'find_and_navigate_combo': Use this for complex queries that ask to *first find a place* and *then get directions* to it.

    Only return the single, most appropriate option's name and nothing else.
    """
)

# We'll create a dictionary of all our individual worker agents
worker_agents = {
    "day_trip_agent": day_trip_agent,
    "foodie_agent": foodie_agent,
    "weekend_guide_agent": weekend_guide_agent,
    "transportation_agent": transportation_agent, # Add the new agent!
}

print("🤖 Agent team assembled for sequential workflows!")

🤖 Agent team assembled for sequential workflows!


In [6]:
# --- Let's Test the Sequential Workflow! ---
import re

async def run_sequential_app():
    queries = [
        "I want to eat the best sushi in Palo Alto.", # Should go to foodie_agent
        "Are there any cool outdoor concerts this weekend?", # Should go to weekend_guide_agent
        "Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station." # Should trigger the COMBO
    ]

    for query in queries:
        print(f"\n{'='*60}\n🗣️ Processing New Query: '{query}'\n{'='*60}")

        # 1. Ask the Router Agent to choose the right agent or workflow
        router_session = await session_service.create_session(app_name=router_agent.name, user_id=my_user_id)
        print("🧠 Asking the router agent to make a decision...")
        chosen_route = await run_agent_query(router_agent, query, router_session, my_user_id, is_router=True)
        chosen_route = chosen_route.strip().replace("'", "")
        print(f"🚦 Router has selected route: '{chosen_route}'")

        # 2. Execute the chosen route
        if chosen_route == 'find_and_navigate_combo':
            print("\n--- Starting Find and Navigate Combo Workflow ---")

            # STEP 2a: Run the foodie_agent first
            foodie_session = await session_service.create_session(app_name=foodie_agent.name, user_id=my_user_id)
            foodie_response = await run_agent_query(foodie_agent, query, foodie_session, my_user_id)

            # STEP 2b: Extract the destination from the first agent's response
            # (This is a simple regex, a more robust solution might use a structured output format)
            match = re.search(r'\*\*(.*?)\*\*', foodie_response)
            if not match:
                print("🚨 Could not determine the restaurant name from the response.")
                continue
            destination = match.group(1)
            print(f"💡 Extracted Destination: {destination}")

            # STEP 2c: Create a new query and run the transportation_agent
            directions_query = f"Give me directions to {destination} from the Palo Alto Caltrain station."
            print(f"\n🗣️ New Query for Transport Agent: '{directions_query}'")
            transport_session = await session_service.create_session(app_name=transportation_agent.name, user_id=my_user_id)
            await run_agent_query(transportation_agent, directions_query, transport_session, my_user_id)

            print("--- Combo Workflow Complete ---")

        elif chosen_route in worker_agents:
            # This is a simple, single-agent route
            worker_agent = worker_agents[chosen_route]
            worker_session = await session_service.create_session(app_name=worker_agent.name, user_id=my_user_id)
            await run_agent_query(worker_agent, query, worker_session, my_user_id)
        else:
            print(f"🚨 Error: Router chose an unknown route: '{chosen_route}'")

await run_sequential_app()


🗣️ Processing New Query: 'I want to eat the best sushi in Palo Alto.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: '9802e11d-443f-4274-9c34-2515c4590170'...
🚦 Router has selected route: 'foodie_agent'

🚀 Running query for agent: 'foodie_agent' in session: '88bd8f8d-63f9-43da-9a7f-724bfff6212c'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""If you're searching for the absolute best sushi experience in Palo Alto, two establishments consistently rise to the top: **Jin Sho** and **Fuki Sushi**.

**Jin Sho** is frequently cited for its exquisite Japanese cuisine and is even noted as being Steve Jobs' favorite restaurant. They are particularly well-regarded for their yellowtail jalapeno sashimi and spicy tuna rolls.

For a traditional Japanese experience with consistent high quality, **Fuki Sushi** is another excellent choice.

Should you be looking for a more value-oriented option witho

If you're searching for the absolute best sushi experience in Palo Alto, two establishments consistently rise to the top: **Jin Sho** and **Fuki Sushi**.

**Jin Sho** is frequently cited for its exquisite Japanese cuisine and is even noted as being Steve Jobs' favorite restaurant. They are particularly well-regarded for their yellowtail jalapeno sashimi and spicy tuna rolls.

For a traditional Japanese experience with consistent high quality, **Fuki Sushi** is another excellent choice.

Should you be looking for a more value-oriented option without compromising on fresh fish and traditional offerings, **Sushi Tomi** comes highly recommended for its cozy atmosphere, generous portions, and reasonable prices. Other notable mentions include **Kanpai Sushi**, praised for its consistent quality, and **Homma's Brown Rice Sushi** for those seeking a good and affordable brown rice sushi option.

--------------------------------------------------


🗣️ Processing New Query: 'Are there any cool outdoor concerts this weekend?'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: '2eeb821b-b329-466d-a77a-feebb71d6bd1'...
🚦 Router has selected route: 'weekend_guide_agent'

🚀 Running query for agent: 'weekend_guide_agent' in session: '4d0f38d9-55b6-4796-8b56-749f22198504'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""This weekend, March 28-29, 2026, offers a variety of outdoor concerts and music festivals across different locations:

*   **Boise, Idaho:** The Treefort Music Fest continues through the weekend until March 29, featuring a diverse lineup of international touring bands and local artists across various venues in downtown Boise.
*   **Brookhaven, Georgia:** The free Brookhaven Cherry Blossom Festival is happening on both Saturday and Sunday, March 28-29. Headliners include The 

This weekend, March 28-29, 2026, offers a variety of outdoor concerts and music festivals across different locations:

*   **Boise, Idaho:** The Treefort Music Fest continues through the weekend until March 29, featuring a diverse lineup of international touring bands and local artists across various venues in downtown Boise.
*   **Brookhaven, Georgia:** The free Brookhaven Cherry Blossom Festival is happening on both Saturday and Sunday, March 28-29. Headliners include The Head and The Heart, Max McNown, Shawn Mullins, and Angie Aparo.
*   **Jackson Hole, Wyoming:** The Rendezvous Music Festival takes place on March 28 at the Jackson Hole Mountain Resort, featuring performances by The Flaming Lips, Umphrey's McGee, and DJ Blake Horstmann. Additionally, there's a free Souls of Mischief concert on Friday, March 27.
*   **Miami, Florida:** Electronic music fans can attend the Ultra Music Festival from March 27-29, showcasing elite DJs and production. Also in Miami, Factory Town Music Week runs from March 25-29, offering dance music artists across five stages.
*   **Washington, D.C.:** On Saturday, March 28, Bloomaroo at The Wharf is a free, family-friendly spring event with live music and a fireworks show.
*   **Knoxville, Tennessee:** The Big Ears Festival, which includes music, a film program, and art exhibitions, runs until March 29.
*   **Palm Beach Gardens, Florida:** The Sounds at Downtown series will feature a free concert with talented local artists on Saturday, March 28.
*   **Baltimore, Maryland:** The TOP OF THE MORNING FESTIVAL is scheduled for March 28, promising music and good vibes.

--------------------------------------------------


🗣️ Processing New Query: 'Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: '33b40ab8-a1ad-4d99-bcde-3848e5e1dc6b'...
🚦 Router has selected route: 'find_and_navigate_combo'

--- Starting Find and Navigate Combo Workflow ---

🚀 Running query for agent: 'foodie_agent' in session: '4236dc16-ab81-468d-acfa-b6a70b755704'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""For the absolute best sushi experience in Palo Alto, I recommend **Jin Sho**. This restaurant has garnered a reputation for high-quality sushi and is even noted as a favorite of Steve Jobs. Jin Sho offers a down-to-earth, traditional Japanese dining experience with excellent sushi, sashimi, and other Japanese tapas. Reviewers often highlight their yellowtail jalapeno sashimi and sp

For the absolute best sushi experience in Palo Alto, I recommend **Jin Sho**. This restaurant has garnered a reputation for high-quality sushi and is even noted as a favorite of Steve Jobs. Jin Sho offers a down-to-earth, traditional Japanese dining experience with excellent sushi, sashimi, and other Japanese tapas. Reviewers often highlight their yellowtail jalapeno sashimi and spicy tuna rolls, and they also offer omakase options for a chef-selected seasonal experience. While considered expensive by Bay Area standards, it's seen as a solid choice for Japanese cuisine lovers given its quality.

To get to **Jin Sho** from the Caltrain station:

Jin Sho is located at 454 California Avenue, Palo Alto, CA 94306. The Palo Alto Caltrain station is at 95 University Ave, Palo Alto, CA 94301.

You can typically reach Jin Sho from the Caltrain station via a short taxi, ride-share, or bus ride. Alternatively, it's a pleasant walk of about 20-25 minutes.

Here are general directions for walking:
1.  From the Caltrain station, head southwest on University Avenue.
2.  Turn right onto Alma Street.
3.  Turn left onto California Avenue. Jin Sho will be on your right.

--------------------------------------------------

💡 Extracted Destination: Jin Sho

🗣️ New Query for Transport Agent: 'Give me directions to Jin Sho from the Palo Alto Caltrain station.'

🚀 Running query for agent: 'transportation_agent' in session: '9cd6abd9-ad83-4f96-abfe-08c7a38eb47a'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""To get to Jin Sho from the Palo Alto Caltrain Station, you can follow these directions:

**Jin Sho's Address:** 454 California Avenue, Palo Alto, California 94306.
**Palo Alto Caltrain Station Address:** 95 University Ave, Palo Alto.

**Walking Directions (approximately 15-20 minutes):**

1.  From the Palo Alto Caltrain Station, head southeast on University Avenue.
2.  Turn right onto Alma Street.
3.  Continue on Alma Street for several blocks.
4.  Turn left onto California Avenue.
5.  Jin Sho will be on your left at 454 California Avenue.

**Driving/Ride-share Directions (approximately 5-10 minutes, dependi

To get to Jin Sho from the Palo Alto Caltrain Station, you can follow these directions:

**Jin Sho's Address:** 454 California Avenue, Palo Alto, California 94306.
**Palo Alto Caltrain Station Address:** 95 University Ave, Palo Alto.

**Walking Directions (approximately 15-20 minutes):**

1.  From the Palo Alto Caltrain Station, head southeast on University Avenue.
2.  Turn right onto Alma Street.
3.  Continue on Alma Street for several blocks.
4.  Turn left onto California Avenue.
5.  Jin Sho will be on your left at 454 California Avenue.

**Driving/Ride-share Directions (approximately 5-10 minutes, depending on traffic):**

1.  From the Palo Alto Caltrain Station, head southeast on University Avenue.
2.  Turn right onto Alma Street.
3.  Continue on Alma Street.
4.  Turn left onto California Avenue.
5.  Jin Sho will be on your left at 454 California Avenue.

Please note that traffic conditions can affect travel time, especially during peak hours. You can use navigation apps like Waze for real-time traffic updates.

--------------------------------------------------

--- Combo Workflow Complete ---


---
### Part 2 (The ADK Way): Multi-Agent Mayhem with `SequentialAgent` 🧠→⛓️→🤖

You've seen how to manually link agents together with custom Python code. It works, but it can get complicated. Now, let's refactor our workflow to use a powerful, built-in ADK feature designed specifically for this: the **`SequentialAgent`**.

The `SequentialAgent` is a *workflow agent*. It's not powered by an LLM itself; instead, its only job is to execute a list of other agents in a strict, predefined order.

The real magic ✨ is how it passes information. The ADK uses a shared `state` dictionary that each agent in the sequence can read from and write to.

**Our New Workflow:**

1.  **Foodie Agent**: Finds the restaurant and saves the name to `state['destination']`.
2.  **Transportation Agent**: Automatically reads `state['destination']` and uses it to find directions.

This means we no longer need custom Python code to extract text or build new queries! The ADK handles the plumbing for us.

```
+-------------------------------+
|  find_and_navigate_agent 🧭   |
| SequentialAgent:             |
| 1. Find destination          |
| 2. Get directions            |
+---------------+---------------+
                |
     +----------+----------+
     |                     |
     v                     v
+----------------+   +------------------------+
| foodie_agent 🍣 |   | transportation_agent 🚗 |
| Finds place     |   | Uses {destination}     |
| Output: 'Jin Sho'|   | Output: Directions     |
+----------------+   +------------------------+

Final Output: 🍣 Restaurant + 🚗 Route
```

In [7]:
# --- Agent Definitions for our Specialist Team (Refactored for Sequential Workflow) ---

# ✨ CHANGE 1: We tell foodie_agent to save its output to the shared state.
# Note the new `output_key` and the more specific instruction.
foodie_agent = Agent(
    name="foodie_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="""You are an expert food critic. Your goal is to find the best restaurant based on a user's request.

    When you recommend a place, you must output *only* the name of the establishment and nothing else.
    For example, if the best sushi is at 'Jin Sho', you should output only: Jin Sho
    """,
    output_key="destination"  # ADK will save the agent's final response to state['destination']
)

# ✨ CHANGE 2: We tell transportation_agent to read from the shared state.
# The `{destination}` placeholder is automatically filled by the ADK from the state.
transportation_agent = Agent(
    name="transportation_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="""You are a navigation assistant. Given a destination, provide clear directions.
    The user wants to go to: {destination}.

    Analyze the user's full original query to find their starting point.
    Then, provide clear directions from that starting point to {destination}.
    """,
)

# ✨ CHANGE 3: Define the SequentialAgent to manage the workflow.
# This agent will run foodie_agent, then transportation_agent, in that exact order.
find_and_navigate_agent = SequentialAgent(
    name="find_and_navigate_agent",
    sub_agents=[foodie_agent, transportation_agent],
    description="A workflow that first finds a location and then provides directions to it."
)

weekend_guide_agent = Agent(
    name="weekend_guide_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="You are a local events guide. Your task is to find interesting events, concerts, festivals, and activities happening on a specific weekend."
)

# --- The Brain of the Operation: The Router Agent ---

# We update the router to know about our new, powerful SequentialAgent.
router_agent = Agent(
    name="router_agent",
    model="gemini-2.5-flash",
    instruction="""
    You are a request router. Your job is to analyze a user's query and decide which of the following agents or workflows is best suited to handle it.
    Do not answer the query yourself, only return the name of the most appropriate choice.

    Available Options:
    - 'foodie_agent': For queries *only* about food, restaurants, or eating.
    - 'weekend_guide_agent': For queries about events, concerts, or activities happening on a specific timeframe like a weekend.
    - 'day_trip_agent': A general planner for any other day trip requests.
    - 'find_and_navigate_agent': Use this for complex queries that ask to *first find a place* and *then get directions* to it.

    Only return the single, most appropriate option's name and nothing else.
    """
)

# We create a dictionary of all our executable agents for easy lookup.
# This now includes our powerful new workflow agent!
worker_agents = {
    "day_trip_agent": day_trip_agent,
    "foodie_agent": foodie_agent,
    "weekend_guide_agent": weekend_guide_agent,
    "find_and_navigate_agent": find_and_navigate_agent, # Add the new sequential agent
}

print("🤖 Agent team assembled with a SequentialAgent workflow!")

🤖 Agent team assembled with a SequentialAgent workflow!


In [8]:
# --- Let's Test the Streamlined Workflow! ---

async def run_sequential_app():
    queries = [
        "I want to eat the best sushi in Palo Alto.", # Should go to foodie_agent
        "Are there any cool outdoor concerts this weekend?", # Should go to weekend_guide_agent
        "Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station." # Should trigger the SequentialAgent
    ]

    for query in queries:
        print(f"\n{'='*60}\n🗣️ Processing New Query: '{query}'\n{'='*60}")

        # 1. Ask the Router Agent to choose the right agent or workflow
        router_session = await session_service.create_session(app_name=router_agent.name, user_id=my_user_id)
        print("🧠 Asking the router agent to make a decision...")
        chosen_route = await run_agent_query(router_agent, query, router_session, my_user_id, is_router=True)
        chosen_route = chosen_route.strip().replace("'", "")
        print(f"🚦 Router has selected route: '{chosen_route}'")

        # 2. Execute the chosen route
        # This logic is now much simpler! The SequentialAgent is treated just like any other worker.
        if chosen_route in worker_agents:
            worker_agent = worker_agents[chosen_route]
            print(f"--- Handing off to {worker_agent.name} ---")
            worker_session = await session_service.create_session(app_name=worker_agent.name, user_id=my_user_id)
            await run_agent_query(worker_agent, query, worker_session, my_user_id)
            print(f"--- {worker_agent.name} Complete ---")
        else:
            print(f"🚨 Error: Router chose an unknown route: '{chosen_route}'")

await run_sequential_app()


🗣️ Processing New Query: 'I want to eat the best sushi in Palo Alto.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: '48adfd80-d531-47f9-ae3c-74de52da9751'...
🚦 Router has selected route: 'foodie_agent'
--- Handing off to foodie_agent ---

🚀 Running query for agent: 'foodie_agent' in session: 'e804d035-b352-4380-bca5-752b0da41441'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text='Jin Sho'
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  retrieval_metadata=RetrievalMetadata(),
  search_entry_point=SearchEntryPoint(
    rendered_content="""<style>
.container {
  align-items: center;
  border-radius: 8px;
  display: flex;
  font-family: Google Sans, Roboto, sans-serif;
  font-size: 14px;
  line-height: 20px;
  padding: 8px 12px;
}
.chip {
  display: inline-block;
  border: solid 1px;
  border-radius: 16px;
  min-width: 14px;
  padding: 5px 16px;
  text-align: center;
  us

Jin Sho

--------------------------------------------------

--- foodie_agent Complete ---

🗣️ Processing New Query: 'Are there any cool outdoor concerts this weekend?'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: '7075e475-ca38-4386-a046-c10b549c8d1e'...
🚦 Router has selected route: 'weekend_guide_agent'
--- Handing off to weekend_guide_agent ---

🚀 Running query for agent: 'weekend_guide_agent' in session: 'b4d31fce-2436-4965-9e19-77f95ad87754'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""This weekend, March 28-29, 2026, offers several exciting outdoor concert and festival options across various locations:

**United States**

*   **Bloomaroo at The Wharf (Washington, DC)**: On Saturday, March 28, 2026, you can enjoy Bloomaroo, a free, family-friendly spring festival at District Pier from 2 PM to 9 PM. This event features live music, art, and a blossom fireworks show.
*   **Jam In the Str

This weekend, March 28-29, 2026, offers several exciting outdoor concert and festival options across various locations:

**United States**

*   **Bloomaroo at The Wharf (Washington, DC)**: On Saturday, March 28, 2026, you can enjoy Bloomaroo, a free, family-friendly spring festival at District Pier from 2 PM to 9 PM. This event features live music, art, and a blossom fireworks show.
*   **Jam In the Streets (Downtown Athens, GA)**: Downtown Athens will host Jam In the Streets on Saturday, March 28, 2026. This college-town music festival includes multiple stages with performances from both world-renowned artists and local college bands, along with vendors and food partners.
*   **Rendezvous Music Festival (Jackson Hole Mountain Resort)**: Head to Jackson Hole on Saturday, March 28, 2026, for the Rendezvous Music Festival. The lineup includes Grammy-winning The Flaming Lips, jam-band Umphrey's McGee, and DJ Blake Horstmann. As a bonus, there's a free Souls of Mischief concert under the Tram on Friday, March 27.
*   **Brookhaven Cherry Blossom Festival (Brookhaven, GA)**: This free two-day festival runs Saturday, March 28, and Sunday, March 29, 2026, at Blackburn Park. Headlining acts include The Head and The Heart and Max McNown, with special guests Shawn Mullins and Angie Aparo also performing.
*   **Sounds at Downtown (Palm Beach Gardens, FL)**: On Saturday, March 28, 2026, from 4 PM to 6:30 PM, you can catch live music at the Green Space in Downtown Palm Beach Gardens. This free concert series features talented local artists.
*   **Busch Gardens Tampa Bay Food and Wine Festival (Tampa, FL)**: The festival continues this weekend with concerts included with park admission. Dylan Scott performs on Saturday, March 28, 2026, at 7 PM, and KANSAS takes the stage on Sunday, March 29, 2026, at 6 PM.

**Other Events with Music**

*   **Beyond Wonderland SoCal (NOS Events Center, Southern California)**: For electronic dance music fans, Beyond Wonderland SoCal is happening March 27-28, 2026, at the NOS Events Center.
*   **TOP OF THE MORNING FESTIVAL (Baltimore, MD)**: On Saturday, March 28, 2026, from 9 AM to 2 PM, the SOUNDSTAGE in Baltimore will host the TOP OF THE MORNING FESTIVAL, offering music and good vibes.

Please note that for specific schedules, ticket information (if applicable), and any potential changes, it is always recommended to check the event organizer's official website.

--------------------------------------------------

--- weekend_guide_agent Complete ---

🗣️ Processing New Query: 'Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: 'e8c41e1a-c1e6-47b9-b644-f7926e5228f5'...
🚦 Router has selected route: 'An error occurred: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {error: {code: 429, message: Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details., status: RESOURCE_EXHAUSTED}}'
🚨 Error: Router chose an unknown route: 'An error occurred: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {error:

### Running the Streamlined App

Notice how much simpler the code below is. There is no longer a special `if chosen_route == 'find_and_navigate_combo':` block with custom logic.

The `find_and_navigate_agent` is now a self-contained unit. We can treat it just like any other agent, hand it the query, and trust the `SequentialAgent` to handle all the internal steps. This makes our main application code much cleaner and easier to read.

---
## Iterative Ideas with `LoopAgent` 🧠→🔁→🤖

Sometimes a task isn't a straight line; it's a loop of refinement. A user might ask for a plan, but have constraints that require checking and re-planning. For this, the ADK provides the **`LoopAgent`**.

The `LoopAgent` executes a sequence of sub-agents repeatedly until a condition is met. This is perfect for workflows involving trial and error, like planning a trip with a tight schedule.

**Our New Workflow: The Perfectionist Planner**

1. **Planner Agent**: Proposes an itinerary (e.g., a museum and a restaurant).
2. **Critic Agent**: Checks the plan against a constraint (e.g., "Is the travel time between these two places less than 30 minutes?").
3. **Refiner Agent**: If the critic finds a problem, this agent takes the feedback and creates a new, improved plan. If the critic is happy, it calls a special `exit_loop` tool to stop the process.

The `LoopAgent` manages this cycle, ensuring we don't get stuck in an infinite loop by setting a `max_iterations` limit.

```
+-------------------------------+
|  iterative_planner_agent 🔁   |
| SequentialAgent:              |
| 1. Propose Plan               |
| 2. Refine in loop (≤ 3 times) |
+---------------+---------------+
                |
     +----------+----------+
     |                     |
     v                     v
+----------------+   +-----------------------+
| planner_agent  |   | refinement_loop 🔁     |
| Propose plan   |   | LoopAgent             |
| e.g., Activity +  | 1. Critic (time check) |
| Restaurant       | 2. Refiner (fix/exit)   |
+----------------+   +-----------------------+

Uses shared state: {current_plan}, {criticism}
Exits when: "Plan is feasible..."

```

In [9]:
# --- Agent Definitions for an Iterative Workflow ---

# A tool to signal that the loop should terminate
COMPLETION_PHRASE = "The plan is feasible and meets all constraints."
def exit_loop(tool_context: ToolContext):
  """Call this function ONLY when the plan is approved, signaling the loop should end."""
  print(f"  [Tool Call] exit_loop triggered by {tool_context.agent_name}")
  tool_context.actions.escalate = True
  return {}

# Agent 1: Proposes an initial plan
planner_agent = Agent(
    name="planner_agent", model="gemini-2.5-flash", tools=[google_search],
    instruction="You are a trip planner. Based on the user's request, propose a single activity and a single restaurant. Output only the names, like: 'Activity: Exploratorium, Restaurant: La Mar'.",
    output_key="current_plan"
)

# Agent 2 (in loop): Critiques the plan
critic_agent = Agent(
    name="critic_agent", model="gemini-2.5-flash", tools=[google_search],
    instruction=f"""You are a logistics expert. Your job is to critique a travel plan. The user has a strict constraint: total travel time must be short.
    Current Plan: {{current_plan}}
    Use your tools to check the travel time between the two locations.
    IF the travel time is over 45 minutes, provide a critique, like: 'This plan is inefficient. Find a restaurant closer to the activity.'
    ELSE, respond with the exact phrase: '{COMPLETION_PHRASE}'""",
    output_key="criticism"
)

# Agent 3 (in loop): Refines the plan or exits
refiner_agent = Agent(
    name="refiner_agent", model="gemini-2.5-flash", tools=[google_search, exit_loop],
    instruction=f"""You are a trip planner, refining a plan based on criticism.
    Original Request: {{session.query}}
    Critique: {{criticism}}
    IF the critique is '{COMPLETION_PHRASE}', you MUST call the 'exit_loop' tool.
    ELSE, generate a NEW plan that addresses the critique. Output only the new plan names, like: 'Activity: de Young Museum, Restaurant: Nopa'.""",
    output_key="current_plan"
)

# ✨ The LoopAgent orchestrates the critique-refine cycle ✨
refinement_loop = LoopAgent(
    name="refinement_loop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=3
)

# ✨ The SequentialAgent puts it all together ✨
iterative_planner_agent = SequentialAgent(
    name="iterative_planner_agent",
    sub_agents=[planner_agent, refinement_loop],
    description="A workflow that iteratively plans and refines a trip to meet constraints."
)

print("🤖 Agent team updated with an iterative LoopAgent workflow!")

🤖 Agent team updated with an iterative LoopAgent workflow!


---
## Parallel Power with `ParallelAgent` 🧠→⚡️→🤖🤖🤖

What if a user wants to find multiple, unrelated things at once? "Find me a museum, a concert, AND a restaurant." Running these searches one by one is slow and inefficient.

Enter the **`ParallelAgent`**. This workflow agent executes a list of sub-agents *concurrently*, dramatically speeding up tasks that can be performed independently.

**Our New Workflow: The Multi-Researcher**

1.  **Parallel Agent**: Simultaneously runs three specialist agents:
    - `MuseumFinderAgent`: Finds a museum.
    - `ConcertFinderAgent`: Finds a concert.
    - `FoodieAgent`: Finds a restaurant.
2.  **Synthesis Agent**: Once all three parallel searches are complete, this final agent gathers the results (which were saved to the shared `state`) and formats them into a single, neat summary for the user.

This pattern lets us get a lot of work done, fast! 🚀

```
+-------------------------------+
|  parallel_planner_agent ⚡     |
| SequentialAgent:              |
| 1. Run parallel research      |
| 2. Synthesize results         |
+---------------+---------------+
                |
     +----------+----------------------+
     |                                 |
     v                                 v
+-------------------------+       +-----------------------------+
| parallel_research_agent ⚡   |   | synthesis_agent 📋          |
| ParallelAgent:              |   | Combine results            |
| - museum_finder_agent 🖼️     |   | Output: Bulleted summary   |
| - concert_finder_agent 🎵    |   +-----------------------------+
| - restaurant_finder_agent 🍽️ |
+-------------------------+

Final Output:
• Museum: XYZ  
• Concert: Artist at Venue  
• Restaurant: ABC
```

In [10]:
# --- Agent Definitions for a Parallel Workflow ---

# Specialist Agent 1
museum_finder_agent = Agent(
    name="museum_finder_agent", model="gemini-2.5-flash", tools=[google_search],
    instruction="You are a museum expert. Find the best museum based on the user's query. Output only the museum's name.",
    output_key="museum_result"
)

# Specialist Agent 2
concert_finder_agent = Agent(
    name="concert_finder_agent", model="gemini-2.5-flash", tools=[google_search],
    instruction="You are an events guide. Find a concert based on the user's query. Output only the concert name and artist.",
    output_key="concert_result"
)

# We can reuse our foodie_agent for the third parallel task!
# Just need to give it a new output_key for this workflow.
# restaurant_finder_agent = foodie_agent.copy(update={"output_key": "restaurant_result"})
restaurant_finder_agent = Agent(
    name="restaurant_finder_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="""You are an expert food critic. Your goal is to find the best restaurant based on a user's request.

    When you recommend a place, you must output *only* the name of the establishment.
    For example, if the best sushi is at 'Jin Sho', you should output only: Jin Sho
    """,
    output_key="restaurant_result" # Set the correct output key for this workflow
)


# ✨ The ParallelAgent runs all three specialists at once ✨
parallel_research_agent = ParallelAgent(
    name="parallel_research_agent",
    sub_agents=[museum_finder_agent, concert_finder_agent, restaurant_finder_agent]
)

# Agent to synthesize the parallel results
synthesis_agent = Agent(
    name="synthesis_agent", model="gemini-2.5-flash",
    instruction="""You are a helpful assistant. Combine the following research results into a clear, bulleted list for the user.
    - Museum: {museum_result}
    - Concert: {concert_result}
    - Restaurant: {restaurant_result}
    """
)

# ✨ The SequentialAgent runs the parallel search, then the synthesis ✨
parallel_planner_agent = SequentialAgent(
    name="parallel_planner_agent",
    sub_agents=[parallel_research_agent, synthesis_agent],
    description="A workflow that finds multiple things in parallel and then summarizes the results."
)

print("🤖 Agent team supercharged with a ParallelAgent workflow!")

🤖 Agent team supercharged with a ParallelAgent workflow!


---
### Final Step: Updating the Router and Running the App

Now we just have one last thing to do: make our `router_agent` aware of these powerful new workflows! We'll add `iterative_planner_agent` and `parallel_planner_agent` to its list of available options.

Then we can run our app with new queries designed to trigger these advanced, multi-agent workflows.

```
                    +---------------------+
                    |    User Query 🗣️     |
                    +----------+----------+
                               |
                               v
                    +---------------------+
                    |   Router Agent 🤖    |
                    |  (Classify Request) |
                    +----------+----------+
                               |
      +-----------+-----------+-----------+-----------+------------+
      |           |           |           |           |            |
      v           v           v           v           v            v
+-------------+  +------------------+  +------------------+  +------------------+  +-----------------+
| foodie_agent|  | find_and_navigate|  | iterative_planner|  | parallel_planner |  | day_trip_agent  |
| 🍣 Food Only |  | 🧭 Seq Workflow   |  | 🔁 Loop Workflow  |  | ⚡ Parallel Tasks |  | 🧳 Basic Plan     |
+-------------+  +------------------+  +------------------+  +------------------+  +-----------------+
```

In [11]:
# --- The ULTIMATE Router Agent --- #

router_agent = Agent(
    name="router_agent",
    model="gemini-2.5-flash",
    instruction="""
    You are a master request router. Your job is to analyze a user's query and decide which of the following agents or workflows is best suited to handle it.
    Do not answer the query yourself, only return the name of the most appropriate choice.

    Available Options:
    - 'foodie_agent': For queries *only* about finding a single food place.
    - 'find_and_navigate_agent': For queries that ask to *first find a place* and *then get directions* to it.
    - 'iterative_planner_agent': For planning a trip with a specific constraint that needs checking, like travel time.
    - 'parallel_planner_agent': For queries that ask to find multiple, independent things at once (e.g., a museum AND a concert AND a restaurant).
    - 'day_trip_agent': A general planner for any other simple day trip requests.

    Only return the single, most appropriate option's name and nothing else.
    """
)

# The master dictionary of all our executable agents and workflows
worker_agents = {
    "day_trip_agent": day_trip_agent,
    "foodie_agent": foodie_agent, # For simple food queries
    "find_and_navigate_agent": find_and_navigate_agent, # Sequential
    "iterative_planner_agent": iterative_planner_agent, # Loop
    "parallel_planner_agent": parallel_planner_agent,   # Parallel
}

# --- Let's Test Everything! ---

async def run_fully_loaded_app():
    queries = [
        # Test Case 1: Simple Sequential Flow
        "Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station.",

        # Test Case 2: Iterative Loop Flow
        "Plan me a day in San Francisco with a museum and a nice dinner, but make sure the travel time between them is very short.",

        # Test Case 3: Parallel Flow
        "Help me plan a trip to SF. I need one museum, one concert, and one great restaurant."
    ]

    for query in queries:
        print(f"\n{'='*60}\n🗣️ Processing New Query: '{query}'\n{'='*60}")

        # 1. Ask the Router Agent to choose the right agent or workflow
        router_session = await session_service.create_session(app_name=router_agent.name, user_id=my_user_id)
        print("🧠 Asking the router agent to make a decision...")
        chosen_route = await run_agent_query(router_agent, query, router_session, my_user_id, is_router=True)
        chosen_route = chosen_route.strip().replace("'", "")
        print(f"🚦 Router has selected route: '{chosen_route}'")

        # 2. Execute the chosen route
        if chosen_route in worker_agents:
            worker_agent = worker_agents[chosen_route]
            print(f"--- Handing off to {worker_agent.name} ---")
            worker_session = await session_service.create_session(app_name=worker_agent.name, user_id=my_user_id)
            await run_agent_query(worker_agent, query, worker_session, my_user_id)
            print(f"--- {worker_agent.name} Complete ---")
        else:
            print(f"🚨 Error: Router chose an unknown route: '{chosen_route}'")

await run_fully_loaded_app()


🗣️ Processing New Query: 'Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: 'b8d519ff-a0c8-4cc8-936f-19413a728cd4'...
🚦 Router has selected route: 'find_and_navigate_agent'
--- Handing off to find_and_navigate_agent ---

🚀 Running query for agent: 'find_and_navigate_agent' in session: '0eb05bf1-f0cb-4c8c-95a1-bd0e19296676'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text='Jin Sho'
    ),
    Part(
      text='To get to Jin Sho from the Palo Alto Caltrain station, head to 454 California Ave, Palo Alto, CA 94306.'
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  grounding_chunks=[
    GroundingChunk(
      web=GroundingChunkWeb(
        domain='waze.com',
        title='waze.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG58rXTJ2NR_1rOR6

You're in luck! Jin Sho, located at 454 California Ave, Palo Alto, CA 94306, has been identified as a top sushi spot in Palo Alto.

Here are directions to Jin Sho from the Palo Alto Caltrain station:

1.  **Exit the Caltrain Station:** Head out of the Palo Alto Caltrain Station onto University Avenue.
2.  **Head South:** From University Avenue, you'll need to head south towards California Avenue.
3.  **Reach California Avenue:** Jin Sho is located at 454 California Avenue.

--------------------------------------------------

--- find_and_navigate_agent Complete ---

🗣️ Processing New Query: 'Plan me a day in San Francisco with a museum and a nice dinner, but make sure the travel time between them is very short.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: 'ca4a13f2-ab4e-4a2f-a836-ce53d02c2b99'...
🚦 Router has selected route: 'iterative_planner_agent'
--- Handing off to iterative_planner_agent ---

🚀 Running query for agent: 'iterative_planner_agent' in session: '68139c28-d8c8-42f4-a3a4-46563128f3dd'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Activity: San Francisco Museum of Modern Art (SFMOMA)
Restaurant: Mourad"""
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  retrieval_metadata=RetrievalMetadata(),
  search_entry_point=SearchEntryPoint(
    rendered_content="""<style>
.container {
  align-items: center;
  border-radius: 8p

EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="The plan is feasible and meets all constraints. SFMOMA is located at 151 Third Street, and Mourad restaurant is at 140 New Montgomery Street, which are in close proximity in San Francisco's South of Market district. The 140 New Montgomery building is described as being close to SFMOMA. This indicates a very short travel time between the two locations."
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  grounding_chunks=[
    GroundingChunk(
      web=GroundingChunkWeb(
        domain='wikipedia.org',
        title='wikipedia.org',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQECiyjsXXq2t_c5fYjEt5_OrFU0gdoyN3uTQB9cjHI0CLnUEU72XIBfrptEL9RpVNchX6p9UFXJx0HIFIwXIY7dsHHTNkAfclbHztcVBsrSGLCOAhF8t5HKRMggczTD0puQ2JLzMlFX8zyRyPc='
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        domain='sfmoma.org',
        title='sfmoma.org',
     

An error occurred: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Multiple tools are supported only when they are all search tools.', 'status': 'INVALID_ARGUMENT'}}

--------------------------------------------------

--- iterative_planner_agent Complete ---

🗣️ Processing New Query: 'Help me plan a trip to SF. I need one museum, one concert, and one great restaurant.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: '67a0db7c-c37c-4f06-a084-07660f6428c1'...
🚦 Router has selected route: 'parallel_planner_agent'
--- Handing off to parallel_planner_agent ---

🚀 Running query for agent: 'parallel_planner_agent' in session: 'c4896a38-e9f9-406b-94f0-57aa96e9c77e'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text='Exploratorium'
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  grounding_chunks=[
    GroundingChunk(
      web=GroundingChunkWeb(
        domain='exploratorium.edu',
        title='exploratorium.edu',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE0Pa9_VsG9A43va6aEjPmfPuNrpAdCunL7KVdo3FYgQ3bPSG

Here are some recommendations for your trip to San Francisco:

*   **Museums:**
    *   Exploratorium
    *   San Francisco Museum of Modern Art (SFMOMA)
*   **Concerts:**
    *   SPRINGSTEEN & E STREET Land of Hope & Dreams American Tour by Bruce Springsteen and E Street Band.
    *   San Francisco Symphony: Philippe Jordan - Symphonie Fantastique & Jean-Yves Thibaudet
*   **Restaurant:**
    *   State Bird Provisions

--------------------------------------------------

--- parallel_planner_agent Complete ---


---
## 🎉 Congratulations! 🎉

You've completed the Enhanced ADK Adventure! You have successfully. Let's review the advanced orchestration patterns you've successfully implemented:

- **The Router Pattern**: You built a master router agent capable of analyzing user intent and delegating tasks to the appropriate specialist agent or workflow.

- **Sequential Workflows**: Using SequentialAgent, you elegantly chained agents together, creating clean, readable code for multi-step tasks without manual data handling.

- **Iterative Refinement**: You constructed a sophisticated feedback loop with LoopAgent, enabling your agents to plan, self-critique, and improve their output until it met specific constraints.

- **Parallel Power**: You maximized speed and efficiency by using ParallelAgent to run multiple research tasks concurrently, later synthesizing the results into a unified response.


```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
